In [1]:
import torch
import triton
import triton.language as tl
DEVICE = torch.device('cuda:0')

In [2]:
@triton.jit
def _seeded_dropout_kernel(
    x_ptr, output_ptr,
    n_elements,
    p,#fp32 [0,1]
    seed, #int32
    BLOCK_SIZE: tl.constexpr
):
  PID = tl.program_id(axis=0)
  offsets = PID * BLOCK_SIZE + tl.arange(0,BLOCK_SIZE)
  mask = offsets < n_elements
  x = tl.load(x_ptr + offsets, mask=mask)
  random = tl.rand(seed, offsets) # random numbers between 0 and 1 of size offsets
  x_keep = random > p
  output = tl.where(x_keep, x/(1 - p), 0.0) #if x_keep is true, put scaled x otherwise put 0
  tl.store(output_ptr+offsets, output, mask=mask)


In [5]:
def seeded_dropout(x, p, seed):
  output = torch.empty_like(x)
  assert x.is_contiguous()
  n_elements = x.numel()
  grid = lambda meta: (triton.cdiv(n_elements, meta["BLOCK_SIZE"]),)
  _seeded_dropout_kernel[grid](
      x, output,
      n_elements,
      p,
      seed,
      BLOCK_SIZE=1024
  )
  return output

In [6]:
x = torch.randn(size=(8,), device=DEVICE)
output1 = seeded_dropout(x, p=0.5, seed=123)
output2 = seeded_dropout(x, p=0.5, seed=123)
output3 = seeded_dropout(x, p=0.5, seed=321)

print(output1, output2, output3, sep="\n")

tensor([ 0.0000,  0.0051,  0.0000,  0.0000,  0.0000, -2.0576,  0.0000,  0.0000],
       device='cuda:0')
tensor([ 0.0000,  0.0051,  0.0000,  0.0000,  0.0000, -2.0576,  0.0000,  0.0000],
       device='cuda:0')
tensor([ 0.0000,  0.0051,  0.0000,  0.0000,  1.4359,  0.0000,  0.0000, -2.7191],
       device='cuda:0')
